In [1]:
import sqlite3

conn = sqlite3.connect(r"C:\Users\Francisco Neto\Desktop\brazilian-ecommerce\database\olist.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tabelas = cursor.fetchall()
for t in tabelas:
    cursor.execute(f"SELECT COUNT(*) FROM {t[0]}")
    print(f"{t[0]}: {cursor.fetchone()[0]} linhas")

conn.close()

pedidos: 99441 linhas
clientes: 99441 linhas
itens: 112650 linhas
produtos: 32951 linhas
pagamentos: 103886 linhas
avaliacoes: 99224 linhas
vendedores: 3095 linhas
categorias: 71 linhas


In [2]:
import sqlite3
import pandas as pd
import plotly.express as px

conn = sqlite3.connect(r"C:\Users\Francisco Neto\Desktop\brazilian-ecommerce\database\olist.db")
print("Conectado!")

Conectado!


In [3]:
df1 = pd.read_sql("""
    SELECT 
        strftime('%Y-%m', order_purchase_timestamp) AS mes,
        COUNT(*) AS total_pedidos
    FROM pedidos
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY mes
    ORDER BY mes
""", conn)

fig = px.line(df1, x="mes", y="total_pedidos", 
              title="Volume de pedidos por mês",
              labels={"mes": "Mês", "total_pedidos": "Pedidos"})
fig.show()

In [4]:
df2 = pd.read_sql("""
    SELECT 
        c.customer_state AS estado,
        COUNT(*) AS atrasos
    FROM pedidos p
    JOIN clientes c ON p.customer_id = c.customer_id
    WHERE p.order_delivered_customer_date > p.order_estimated_delivery_date
    AND p.order_delivered_customer_date IS NOT NULL
    GROUP BY estado
    ORDER BY atrasos DESC
    LIMIT 10
""", conn)

fig = px.bar(df2, x="estado", y="atrasos",
             title="Top 10 estados com mais atrasos",
             labels={"estado": "Estado", "atrasos": "Qtd atrasos"})
fig.show()

In [5]:
df3 = pd.read_sql("""
    SELECT 
        cat.product_category_name_english AS categoria,
        COUNT(*) AS vendas
    FROM itens i
    JOIN produtos p ON i.product_id = p.product_id
    JOIN categorias cat ON p.product_category_name = cat.product_category_name
    GROUP BY categoria
    ORDER BY vendas DESC
    LIMIT 10
""", conn)

fig = px.bar(df3, x="vendas", y="categoria", orientation="h",
             title="Top 10 categorias mais vendidas",
             labels={"vendas": "Vendas", "categoria": "Categoria"})
fig.show()

In [6]:
df4 = pd.read_sql("""
    SELECT 
        r.review_score AS nota,
        AVG(julianday(p.order_delivered_customer_date) - 
            julianday(p.order_purchase_timestamp)) AS dias_entrega
    FROM pedidos p
    JOIN avaliacoes r ON p.order_id = r.order_id
    WHERE p.order_delivered_customer_date IS NOT NULL
    GROUP BY nota
    ORDER BY nota
""", conn)

fig = px.bar(df4, x="nota", y="dias_entrega",
             title="Tempo médio de entrega por nota",
             labels={"nota": "Nota", "dias_entrega": "Dias até entrega"})
fig.show()